# Card 2024 · speech T15

## Prepare the data

Run the commands below from the repository root with `bci-repro` active.

### 1. Download

Download `t15_copyTask_neuralData.zip` from [Dryad](https://doi.org/10.5061/dryad.dncjsxm85). Create `studies/20240815_card/data/raw/dryad_dncjsxm85/` and save the download as `studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_copyTask_neuralData.zip`.

### 2. Extract

Extract the archive into `studies/20240815_card/data/extracted/dryad_dncjsxm85/`.

```text
python -m zipfile -e "studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_copyTask_neuralData.zip" "studies/20240815_card/data/extracted/dryad_dncjsxm85"
```

After extraction, check for `studies/20240815_card/data/extracted/dryad_dncjsxm85/hdf5_data_final/`.

### 3. Prepare the model input

The trainer reads the released HDF5 directly from `studies/20240815_card/data/extracted/dryad_dncjsxm85/hdf5_data_final/`. This directory is the HDF5 input for training and inference.

Set up the upstream trainer. For inference or logit export, copy `studies/20240815_card/open_source_code_for_reference/nejm-brain-to-text/data/t15_copyTaskData_description.csv` into `studies/20240815_card/data/extracted/dryad_dncjsxm85/`.

```text
python shared/tools/setup_upstream.py --study 20240815_card
```

## Read the archived results

Figure 3a uses the validation trials and day-index-corrected decode from the P4 GRU. Figure 2f uses the BiGRU comparison. The rapid-calibration results are a separate archive.

This walkthrough uses the checked-in archive and takes no data download.

In [1]:
import csv
import json
import sys
from pathlib import Path

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "figures/reproduction_benchmark.yaml").is_file())
sys.path.insert(0, str(root / "figures/code"))
from figure_paths import PUBLISHED, load_benchmark

benchmark = load_benchmark()

def read_rows(path):
    """Read the archived CSV rows in their original order."""
    with (root / path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))

## From a trial or day to the reported result

In [2]:
arm = "Card 2024 (T15)"
from build_session_per_vs_wer import build, read_tsv, word_ed

source = benchmark["figure3_per_sources"][arm]
rows = read_rows(source["greedy"])
decoded = read_tsv(root / source["decode"])
trial = rows[0]
reference, hypothesis = decoded[0]
print(f"{trial['session']} · {reference}")
print(f"Word hypothesis: {hypothesis}")
edits, words = word_ed(reference, hypothesis)
print(f"Trial word edits: {edits}/{words}")
result = build(arm, root / source["greedy"], root / source["decode"])
day = result["sessions"][0]
print(f"{day['session']}: {day['n']} trials, PER {day['per']:.2f}%, WER {day['wer']:.2f}%")
print(f"{len(result['sessions'])} sessions, {len(rows)} evaluated trials; pooled {result['pooled']}")

t15.2023.08.13 · You can see the code at this point as well.
Word hypothesis: you can see the code at this point as well
Trial word edits: 0/10
Card 2024 (T15)   : 41 sessions | pooled PER 10.31% | WER  7.02%
t15.2023.08.13: 35 trials, PER 6.44%, WER 3.48%
41 sessions, 1426 evaluated trials; pooled {'per': 10.31, 'wer': 7.02}


## Continue the full reproduction

If you use the released weights, download `t15_pretrained_rnn_baseline.zip`, save it at `studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_pretrained_rnn_baseline.zip`, and extract it with this command.

```text
python -m zipfile -e "studies/20240815_card/data/raw/dryad_dncjsxm85/dryad_dncjsxm85__t15_pretrained_rnn_baseline.zip" "studies/20240815_card/data/checkpoints/dryad_dncjsxm85"
```

The checkpoint directory is `studies/20240815_card/data/checkpoints/dryad_dncjsxm85/t15_pretrained_rnn_baseline/`.

For a separate data disk, set `BCI_DATA_ROOT` to the directory that contains the study folders and use the same layout under that root.

See the [evaluation and input guide](docs/20261124_evaluation_protocol/README.md) and [code guide](code/README.md).

**Public commands** [`train.py`](code/train.py), [`infer.py`](code/infer.py), [`evaluate.py`](code/evaluate.py), [`decode.py`](code/decode.py). Run a command with `--help` to see its arguments.